[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/06_Humedad_Suelo/01_humedad_suelo.ipynb)

# Humedad de suelo: estado, percentiles y anomalia (SMAP)
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [ ]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

In [ ]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "iotapp7"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [ ]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

In [ ]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

In [ ]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

# M6. Humedad de suelo: estado actual, percentiles y anomalía
**Dimensiones:** *ocurrencia* (meses secos), *intensidad* (percentil/anomalía),
*duración* (racha de meses secos), *tendencia* (serie mensual).

* **Primario:** `NASA/SMAP/SPL4SMGP/008` — humedad radicular (`sm_rootzone`, fracción 0–1),
  percentil (`sm_rootzone_pctl`, 0–100) y anomalía (`sm_surface_anomaly`) del satélite SMAP.
  **Período nativo: 2015–actual** (se declara explícitamente; no se rellenan años sin dato).
* **Alternativa para la ventana completa de 30 años:** capas de suelo de ERA5-Land
  (`volumetric_soil_water_layer_2..4`, 1950+, ~11 km) — comentada en la celda.
* **Ojo:** SMAP tiene ~11 km de resolución y ~3 días de repetición; describe zonas
  extensas, no el suelo de una parcela.

<details>
<summary><b>📐 Cómo se lee la humedad radicular</b> — clic para ver</summary>
<br>

* **% volumétrico:** cuánta fracción del suelo está ocupada por agua (0 = seco, ~40–55% = saturado
  según textura; ver `M6.2` y el perfil de suelo del punto de interés).
* **Percentil (0–100):** dónde está el valor respecto a su propio histórico en ese mes:
  < 10 = entre los 10% más secos registrados (alerta); > 90 = entre los 10% más húmedos.
* **Anomalía:** desviación respecto a la media histórica (valores negativos = más seco de lo normal).

</details>

In [ ]:
# ============================================================
# M6.1 Humedad radicular: serie mensual y umbrales (SMAP L4)
# ============================================================

# --- Primario: SMAP L4 (2015+) -----------------------------------------------
co_smap = (
    ee.ImageCollection("NASA/SMAP/SPL4SMGP/008")
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filterBounds(roi)
)

# --- Alternativas (comentar primaria y descomentar) ---------------------------
# co_smap = ee.ImageCollection("NASA/SMAP/SPL4SMGP/007")   # versión anterior (deprecada)
# Para cubrir 1996–2026 completo: ERA5-Land capa 2 (0–28 cm), fracción 0–1:
# co_smap = (
#     ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
#     .filterDate(FECHA_INICIO, FECHA_FIN)
#     .filterBounds(roi)
#     .map(lambda i: i.select("volumetric_soil_water_layer_2").rename("sm_rootzone"))
# )
# NOTA: si se usa la alternativa, en M6.2 reemplazar "sm_rootzone_pctl" y
# "sm_surface_anomaly" por variables derivadas (percentil/anomalía calculados).

# Años con dato real (SMAP inicia en 2015): se declara, no se inventa
_fechas = pd.to_datetime(
    co_smap.aggregate_array("system:time_start").getInfo(), unit="ms"
)
anios_smap = sorted({int(a) for a in _fechas.year.unique()})
anios_smap = [y for y in anios_smap if ANIO_INI <= y <= ANIOS_COMPLETOS[-1]]
print(f"Ventana CONFIG: {ANIO_INI}-{ANIOS_COMPLETOS[-1]} (regla 30 años)")
print(f"SMAP con dato: {anios_smap[0]}-{anios_smap[-1]} ({len(anios_smap)} años)")

# --- Serie mensual (promedio del AOI por mes) --------------------------------
imgs, nombres = [], []
for y in anios_smap:
    for m in MESES:
        co_m = (
            co_smap
            .filter(ee.Filter.calendarRange(y, y, "year"))
            .filter(ee.Filter.calendarRange(m, m, "month"))
        )
        imgs.append(co_m.mean().select("sm_rootzone").rename(f"{y}-{m:02d}"))
        nombres.append(f"{y}-{m:02d}")

d_sm = reducir_bandas(imgs, nombres, ESCALAS["clima"], tam=12)
serie_sm = pd.Series(
    {pd.Timestamp(k): v for k, v in d_sm.items() if v is not None}
).sort_index()

# Escala: si es fracción 0-1 -> convertir a %
if len(serie_sm) and np.nanmedian(serie_sm.values) <= 1.5:
    serie_sm = 100 * serie_sm

# --- Umbrales de su propio histórico ------------------------------------------
P10, P50, P90 = np.percentile(serie_sm.values, [10, 50, 90])
df_sm = serie_sm.to_frame("rootzone_pct")
df_sm["seco"] = df_sm["rootzone_pct"] < P10

# Racha máxima de meses secos (función max_racha definida en M5.1)
racha_max_sm = max_racha(list(df_sm["seco"]))

print(f"Humedad radicular: media {serie_sm.mean():.1f}% | "
      f"P10 {P10:.1f}% | P50 {P50:.1f}% | P90 {P90:.1f}%")
print(f"Meses por debajo del P10: {int(df_sm['seco'].sum())} de {len(df_sm)} "
      f"({100 * df_sm['seco'].mean():.0f}%) | máxima racha seca: {racha_max_sm} meses")

display(df_sm.round(1).head())

# --- Gráfico ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(df_sm.index, df_sm["rootzone_pct"], color="#1565c0", lw=1.2, label="Humedad radicular")
ax.axhline(P90, color="#1976d2", ls=":", lw=1, label=f"P90 ({P90:.0f}%)")
ax.axhline(P50, color="gray", ls="--", lw=1, label=f"P50 ({P50:.0f}%)")
ax.axhline(P10, color="#d32f2f", ls=":", lw=1.5, label=f"P10 ({P10:.0f}%) = umbral seco")
ax.scatter(df_sm.index[df_sm["seco"]], df_sm.loc[df_sm["seco"], "rootzone_pct"],
           c="#d32f2f", s=12, zorder=3, label="meses secos")
ax.set_title("Humedad radicular mensual (%): estado vs. su propio histórico")
ax.set_ylabel("% de humedad en la zona radicular")
ax.legend(ncol=4, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# M6.2 Mapas: percentil histórico, último estado y anomalía
# ============================================================

# Percentil histórico de la humedad radicular (0-100)
pctl_media = co_smap.select("sm_rootzone_pctl").mean().rename("pctl")

# Último estado disponible (fecha real del último dato SMAP)
ultima = ee.Image(co_smap.sort("system:time_start").last())
fecha_ultima = ee.Date(ultima.get("system:time_start")).format("yyyy-MM-dd").getInfo()

# Anomalía promedio de las últimas 4 semanas
anom_4sem = (
    co_smap
    .filterDate(ee.Date(FECHA_FIN).advance(-28, "day"), FECHA_FIN)
    .mean()
    .select("sm_surface_anomaly")
)

pctl_aoi = promediar_aoi(pctl_media, "pctl", ESCALAS["clima"])
anom_aoi = promediar_aoi(anom_4sem, "sm_surface_anomaly", ESCALAS["clima"])
print(f"Último dato SMAP disponible: {fecha_ultima}")
print(f"Percentil medio del AOI: {pctl_aoi:.0f}/100  |  "
      f"Anomalía promedio (últimas 4 semanas): {anom_aoi:+.3f} m³/m³")

vis_pctl = {"min": 0, "max": 100,
            "palette": ["d73027", "fc8d59", "fee090", "d9ef8b", "91cf60", "1a9850"]}
vis_anom = {"min": -0.15, "max": 0.15,
            "palette": ["a50026", "f46d43", "fddbc7", "f7f7f7", "d1e5f0", "4393c3", "053061"]}

Map_m6 = geemap.Map()
Map_m6.centerObject(roi, ZONA["zoom"])
Map_m6.add_basemap("HYBRID")
Map_m6.addLayer(pctl_media, vis_pctl, "Percentil histórico de humedad radicular", False)
Map_m6.addLayer(
    ultima.select("sm_rootzone").multiply(100),
    {"min": 0, "max": 100, "palette": vis_pctl["palette"]},
    f"Último estado ({fecha_ultima}, %)",
)
Map_m6.addLayer(anom_4sem, vis_anom, "Anomalía últimas 4 semanas (m³/m³)", False)
Map_m6.addLayer(borde_aoi(), {}, "AOI")
Map_m6.add_colorbar(vis_pctl, label="Percentil (0-100)",
                    orientation="horizontal", position="bottomright",
                    layer_name="Percentil de humedad")
Map_m6

## Interpretación de M6
* **Percentil bajo (< 10)** en la zona radicular = condición entre las más secas del histórico:
  es la señal accionable para riego o alerta temprana (complementa el SPI de M5, que mide lluvia).
* **Anomalía negativa persistente** varias semanas = sequía de suelo en desarrollo aunque llueva
  poco; la atmósfera y el suelo se desacoplan.
* **Duración:** la máxima racha de meses por debajo del P10 (impresa arriba) mide eventos largos.
* La ventana con dato real de SMAP es **2015–actual**; para 30 años completos usar la
  alternativa ERA5-Land comentada en M6.1.

**Preguntas guía**
1. ¿Los meses que ustedes marcan como "secos en el campo" caen bajo el P10 en la serie?
2. ¿La anomalía reciente coincide con lo que miden sus sensores de humedad de suelo (CLIMBAT/PACHASOL)?
3. ¿Qué años con menor NDVI (M7) muestran aquí los percentiles más bajos?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
